# Daily order allocation, warehouse expansion and outage risk

Run the cells from top to bottom with the **Python (gurobi-env)** kernel. The input workbook is `data/Supply chain logisitcs problem.xlsx`. Costs are in US dollars. Capacity is measured in orders per day. An order counts toward throughput when a warehouse schedules it for dispatch that day.

| Model input | Data source and calculation |
|---|---|
| Orders $I$, quantity $q_i$ and weight $g_i$ | Take DTD and DTP orders from `OrderList`. Use the recorded quantity and weight. All orders are available at the start of the planning day. |
| Daily capacity $B_w$ and storage cost per unit $h_w$ | Use the daily order limit in `WhCapacities` and the cost per unit in `WhCosts`. |
| Eligible shipping options $A_i$ | Match products, customers and origin ports using `ProductsPerPlant`, `VmiCustomers` and `PlantPorts`. Then match the destination, service level and weight band in `FreightRates`. Each option identifies a warehouse, an origin port and a quoted rate row. |
| Cost $c_{ia}$ of shipping order $i$ using option $a$ | Price each order separately in all three questions. Storage cost is $q_i h_w$. For air freight, use the higher of the minimum charge and the order weight times the rate. For ground freight, use the single-vehicle quote for the applicable weight band. |
| Reserved slots $R_w$, extra slots $\Delta_w$ and warehouse limit $K$ | Set $R_w=0$ in the baseline. In the reservation scenario, count CRF orders at each historical warehouse. For expansion, add $\Delta_w=100$ daily slots at each selected warehouse and set $K=2$. |

[Dataset documentation](https://doi.org/10.17633/rd.brunel.7558679.v2) · [Freight cost calculation, section 4.2](https://bura.brunel.ac.uk/bitstream/2438/20185/5/Preprint.pdf)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display

filename = "Supply chain logisitcs problem.xlsx"
paths = [Path("data") / filename, Path("group_project/data") / filename, Path.home() / "Downloads" / filename]
workbook = next(p for p in paths if p.is_file())
sheets = pd.read_excel(workbook, sheet_name=None, engine="openpyxl")
for sheet in sheets.values():
    sheet.columns = sheet.columns.str.strip()

orders_all = sheets["OrderList"].rename(columns={
    "Order ID": "order", "Product ID": "product", "Customer": "customer",
    "Plant Code": "historical_plant", "Carrier": "historical_carrier", "Destination Port": "destination",
    "Service Level": "service", "Unit quantity": "units", "Weight": "weight", "TPT": "historical_days"
})
orders_all["order"] = orders_all["order"].astype(str)
assert orders_all["order"].is_unique
orders = orders_all.loc[orders_all.service.isin(["DTD", "DTP"])].copy()
capacity = sheets["WhCapacities"].set_index("Plant ID")["Daily Capacity"].sort_index().astype(int)
storage = sheets["WhCosts"].set_index("WH")["Cost/unit"]
crf_count = (orders_all.loc[orders_all.service.eq("CRF"), "historical_plant"]
             .value_counts().reindex(capacity.index, fill_value=0))
assert (crf_count <= capacity).all()

products = sheets["ProductsPerPlant"].rename(columns={"Plant Code": "plant", "Product ID": "product"})
ports = sheets["PlantPorts"].rename(columns={"Plant Code": "plant", "Port": "port"})
vmi = sheets["VmiCustomers"].rename(columns={"Plant Code": "plant", "Customers": "customer"})
rates = sheets["FreightRates"].drop_duplicates().rename(columns={
    "Carrier": "carrier", "orig_port_cd": "port", "dest_port_cd": "destination", "svc_cd": "service",
    "minm_wgh_qty": "low", "max_wgh_qty": "high", "minimum cost": "minimum",
    "mode_dsc": "mode", "tpt_day_cnt": "days"
}).copy()
rates["quote"] = rates.index + 2  # Original worksheet row number, including the header.
rates["mode"] = rates["mode"].str.strip()
assert set(rates["mode"]) == {"AIR", "GROUND"}

# A warehouse with a VMI list can serve only the customers on that list.
routes = (orders[["order", "product", "customer", "destination", "service", "units", "weight", "historical_days"]]
          .merge(products, on="product").merge(ports, on="plant")
          .merge(rates, on=["port", "destination", "service"]))
routes = routes.loc[routes.weight.between(routes.low, routes.high, inclusive="both")].copy()
allowed_pairs = pd.MultiIndex.from_frame(vmi[["plant", "customer"]])
routes["vmi_ok"] = (~routes.plant.isin(vmi.plant)
                    | pd.MultiIndex.from_frame(routes[["plant", "customer"]]).isin(allowed_pairs))
routes["warehouse_cost"] = routes.units * routes.plant.map(storage)
routes["freight"] = np.where(routes["mode"].eq("AIR"),
                             np.maximum(routes.minimum, routes.weight * routes.rate), routes.rate)
routes["cost"] = routes.warehouse_cost + routes.freight
assert routes["cost"].notna().all()
# Keep all quotes until the scenario filters are applied, then keep the cheapest option per order and warehouse.
routes = routes.sort_values(["cost", "days", "quote", "plant"], kind="stable")
eligible = routes.loc[routes.vmi_ok]
def lowest_by_plant(options):
    return options.drop_duplicates(["order", "plant"]).reset_index(drop=True)

base_options = lowest_by_plant(eligible)
display(pd.DataFrame([{
    "DTD/DTP orders": len(orders), "Orders with an eligible option": eligible.order.nunique(),
    "Original quotes": len(sheets["FreightRates"]), "Distinct quotes": len(rates),
    "Zero-weight orders": orders.weight.eq(0).sum(), "Order-warehouse pairs": len(base_options)
}]))

,DTD/DTP orders,Orders with an eligible option,Original quotes,Distinct quotes,Zero-weight orders,Order-warehouse pairs
0,8361,8361,1540,1537,2,10046


## Question 1. How many orders can we process at current capacity, and at what cost?

We first find the maximum flow, then find the minimum-cost flow at that volume. The variable $x_{ia}$ records whether order $i$ uses shipping option $a$. Each order uses one slot. With integer capacities, this network admits a whole-order optimum, so we can use continuous variables and the simplex method.

$$\sum_{a\in A_i}x_{ia}\le 1,\qquad
\sum_{i,a:\,w(a)=w} x_{ia}\le B_w-R_w,\qquad x_{ia}\ge 0.$$

First maximize the daily order count $N=\sum_{i,a}x_{ia}$. Then fix $N=N^*$ and minimize total operating cost $\sum_{i,a}c_{ia}x_{ia}$.

In [2]:
env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.setParam("Method", 1)  # Dual simplex
env.setParam("Threads", 2)
env.setParam("MIPGap", 0)
env.setParam("MIPGapAbs", 0)
env.start()

def add_assignment(model, options, caps, *, required=False, name="assign"):
    # Each order-warehouse pair has one least-cost option. A fixed batch requires every order ID exactly once.
    x = model.addVars(list(zip(options.order, options.plant)), ub=1, name=name)
    for i in options.order.unique():
        count_i = x.sum(i, "*")
        model.addConstr(count_i == 1 if required else count_i <= 1)
    limits = model.addConstrs((x.sum("*", w) <= caps[w] for w in caps), name=name + "_capacity")
    cost = gp.LinExpr(options.cost.to_list(), list(x.values()))
    return x, cost, limits

def optimum(model, objective, sense):
    model.setObjective(objective, sense)
    model.optimize()
    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Gurobi did not find an optimum (status {model.Status}).")
    return model.ObjVal

def solve_assignment(options, caps, *, throughput_only=False):
    if (caps < 0).any():
        raise ValueError("Available warehouse capacity must be nonnegative.")
    with gp.Model(env=env) as model:
        x, cost, limits = add_assignment(model, options, caps.to_dict())
        count = x.sum()
        maximum = round(optimum(model, count, GRB.MAXIMIZE))
        if throughput_only:
            return {"maximum": maximum}
        throughput_dual = pd.Series(model.getAttr("Pi", limits))
        model.addConstr(count == maximum)
        minimum = optimum(model, cost, GRB.MINIMIZE)
        values = np.array(model.getAttr("X", list(x.values())))
        assert np.allclose(values, values.round(), atol=1e-6, rtol=0)
        chosen = options.loc[values > 0.5].copy()
        load = chosen.plant.value_counts().reindex(caps.index, fill_value=0)
        assert len(chosen) == maximum and chosen.order.is_unique and (load <= caps).all()
        assert np.isclose(chosen.cost.sum(), minimum, atol=1e-5, rtol=0)
        return {"maximum": maximum, "cost": minimum, "chosen": chosen, "load": load,
                "throughput_dual": throughput_dual, "cost_dual": pd.Series(model.getAttr("Pi", limits))}


In [3]:
base = solve_assignment(base_options, capacity)
without_vmi = solve_assignment(lowest_by_plant(routes), capacity)
assert (len(orders), eligible.order.nunique(), base["maximum"], without_vmi["maximum"]) == (8361, 8361, 2291, 2636)
assert np.isclose(base["cost"], 743755.6256404707, atol=0.01, rtol=0)

def baseline_row(label, demand, options, result):
    return {"Scenario": label, "Orders in pool": demand, "Orders with an eligible option": options.order.nunique(),
            "Maximum orders": result["maximum"], "Minimum operating cost (USD)": round(result["cost"], 2)}

checks = [baseline_row("Baseline", len(orders), base_options, base),
          baseline_row("VMI restrictions lifted", len(orders), routes, without_vmi)]
for label, demand, options, caps in [
    ("Transit time capped at historical value", len(orders), eligible.loc[eligible.days <= eligible.historical_days], capacity),
    ("Zero-weight orders excluded", int(orders.weight.gt(0).sum()), eligible.loc[eligible.weight > 0], capacity),
    ("CRF slots reserved by historical warehouse", len(orders), eligible, capacity - crf_count)
]:
    result = solve_assignment(lowest_by_plant(options), caps)
    checks.append(baseline_row(label, demand, options, result))
baseline_checks = pd.DataFrame(checks).set_index("Scenario")
display(baseline_checks)
order_plan = base["chosen"][["order", "plant", "port", "carrier", "quote", "days", "warehouse_cost", "freight", "cost"]]
display(order_plan.head(3).round(2).rename(columns={"order":"Order ID","plant":"Warehouse","port":"Port","carrier":"Carrier",
                                                    "quote":"Quote row","days":"Transit days","warehouse_cost":"Storage cost",
                                                    "freight":"Freight cost","cost":"Total cost"}))
print(f"The baseline can process {base['maximum']:,} orders for a minimum cost of ${base['cost']:,.2f}. "
      "The full assignment is in order_plan.")

,Orders in pool,Orders with an eligible option,Maximum orders,Minimum operating cost (USD)
Scenario,,,,
Baseline,8361,8361,2291,743755.63
VMI restrictions lifted,8361,8361,2636,805806.24
Transit time capped at historical value,8361,8361,2277,730919.36
Zero-weight orders excluded,8359,8359,2290,743600.70
CRF slots reserved by historical warehouse,8361,8361,1578,637347.78


,Order ID,Warehouse,Port,Carrier,Quote row,Transit days,Storage cost,Freight cost,Total cost
0,1447368565.7,PLANT04,PORT05,V444_5,210,3,106.27,5.81,112.08
1,1447161166.7,PLANT04,PORT05,V444_5,210,3,107.13,5.81,112.93
2,1447220210.7,PLANT04,PORT05,V444_5,210,3,107.55,5.81,113.36


The baseline can process 2,291 orders for a minimum cost of $743,755.63. The full assignment is in order_plan.


## Question 2. Which warehouses should we expand to increase throughput and save on the same orders?

Select up to two warehouses. Each gets 100 extra slots per day. The binary variable $y_w$ equals 1 if warehouse $w$ expands. With zero reserved slots, its daily capacity is $B_w+100y_w$, and $\sum_w y_w\le 2$.

- **Additional orders per day.** Use the full order pool. Subtract maximum throughput before expansion from maximum throughput after expansion.
- **Savings on the same orders.** Keep every order ID selected in the Question 1 baseline. Warehouses and shipping routes can change. Subtract the minimum cost of serving this batch after expansion from its minimum cost before expansion.

First test each warehouse on its own. Then use a mixed-integer linear model to choose the combination with the highest throughput. If several combinations tie, choose the one with the lowest cost for the fixed batch.

In [4]:
EXTRA = 100
SITE_LIMIT = 2
N0 = base["maximum"]
base_ids = frozenset(order_plan.order)
fixed_options = base_options.loc[base_options.order.isin(base_ids)]
assert len(base_ids) == fixed_options.order.nunique() == N0

def fixed_batch_cost(caps):
    result = solve_assignment(fixed_options, caps)
    assert frozenset(result["chosen"].order) == base_ids
    return result["cost"]

before_cost = fixed_batch_cost(capacity)
assert np.isclose(before_cost, base["cost"], atol=1e-5, rtol=0)

single_rows = []
for plant in capacity.index:
    caps = capacity.copy()
    caps.loc[plant] += EXTRA
    maximum = solve_assignment(base_options, caps, throughput_only=True)["maximum"]
    saving = before_cost - fixed_batch_cost(caps)
    assert maximum >= N0 and saving >= -1e-5
    single_rows.append({"Warehouse": plant, "Current capacity": capacity[plant], "Baseline orders assigned": base["load"][plant],
                        "Additional orders per day": maximum - N0, "Fixed-batch savings (USD)": max(0.0, saving)})
single_expansion = (pd.DataFrame(single_rows).set_index("Warehouse")
                    .sort_values(["Additional orders per day", "Fixed-batch savings (USD)"], ascending=False, kind="stable"))
display(single_expansion.style.format({"Fixed-batch savings (USD)": "{:,.2f}"}))


,Current capacity,Baseline orders assigned,Additional orders per day,Fixed-batch savings (USD)
Warehouse,,,,
PLANT02,138,138,100,126.93
PLANT03,1013,1013,100,0.00
PLANT12,209,209,65,"8,999.06"
PLANT10,118,118,65,0.00
PLANT09,11,11,43,0.00
PLANT01,1070,0,0,0.00
PLANT04,554,375,0,0.00
PLANT05,385,179,0,0.00
PLANT06,49,0,0,0.00


In [5]:
# Both comparisons use the same expansion sites. Each uses the full capacity in a separate assignment.
with gp.Model(env=env) as model:
    y = model.addVars(capacity.index.to_list(), vtype=GRB.BINARY, name="expand")
    model.addConstr(y.sum() <= SITE_LIMIT)
    caps_y = {w: int(capacity[w]) + EXTRA * y[w] for w in capacity.index}

    x, _, _ = add_assignment(model, base_options, caps_y)
    maximum = round(optimum(model, x.sum(), GRB.MAXIMIZE))
    model.addConstr(x.sum() == maximum)
    _, fixed_cost, _ = add_assignment(model, fixed_options, caps_y, required=True, name="fixed")
    minimum = optimum(model, fixed_cost, GRB.MINIMIZE)
    sites = sorted(w for w in capacity.index if y[w].X > 0.5)

expanded_capacity = capacity.copy()
expanded_capacity.loc[sites] += EXTRA
expanded = solve_assignment(base_options, expanded_capacity, throughput_only=True)
after_cost = fixed_batch_cost(expanded_capacity)
assert expanded["maximum"] == maximum and np.isclose(after_cost, minimum, atol=1e-5, rtol=0)
gain = maximum - N0
saving = before_cost - after_cost  # Calculate savings before rounding to cents for display.
sites_text = " + ".join(sites)
comparison = pd.DataFrame([
    ["Current capacity", N0, 0, before_cost, 0.0],
    [sites_text + " expansion", maximum, gain, after_cost, saving]
], columns=["Capacity plan", "Maximum orders per day", "Additional orders per day",
            "Minimum fixed-batch cost (USD)", "Fixed-batch savings (USD)"]).set_index("Capacity plan")
display(comparison.style.format({"Minimum fixed-batch cost (USD)": "{:,.2f}", "Fixed-batch savings (USD)": "{:,.2f}"}))
print(f"Adding {EXTRA} slots per warehouse at {sites_text} raises daily throughput by {gain:,} orders. "
      f"Serving the same {N0:,} baseline orders saves ${saving:,.2f}.")


,Maximum orders per day,Additional orders per day,Minimum fixed-batch cost (USD),Fixed-batch savings (USD)
Capacity plan,,,,
Current capacity,2291,0,"743,755.63",0.00
PLANT02 + PLANT03 expansion,2491,200,"743,628.69",126.93


Adding 100 slots per warehouse at PLANT02 + PLANT03 raises daily throughput by 200 orders. Serving the same 2,291 baseline orders saves $126.93.


## Question 3. Which outages cause the largest losses, and where should we expand in advance?

### 3.1 Which outage causes the largest loss at current capacity?

Close one warehouse or carrier at a time. Reassign orders from the Question 1 order pool. For each outage, subtract the remaining maximum throughput from the normal-day maximum.

In [6]:
plants = capacity.index.to_list()
carriers = sorted(rates.carrier.unique())

# For a warehouse outage, remove its shipping options. For a carrier outage, remove its quotes before choosing alternatives.
scenario_options = {"Normal operations": base_options}
for plant in plants:
    scenario_options[plant] = base_options.loc[base_options.plant != plant]
for carrier in carriers:
    scenario_options[carrier] = lowest_by_plant(eligible.loc[eligible.carrier != carrier])

outage_rows = []
for kind, nodes in [("Warehouse", plants), ("Carrier", carriers)]:
    for node in nodes:
        served = solve_assignment(scenario_options[node], capacity, throughput_only=True)["maximum"]
        outage_rows.append({"Type": kind, "Outage at": node, "Orders during outage": served,
                            "Drop in daily orders": N0 - served, "Throughput loss (%)": 100 * (N0 - served) / N0})

disruptions = pd.DataFrame(outage_rows).sort_values(["Drop in daily orders", "Outage at"], ascending=[False, True])
print(f"Current capacity supports {N0:,} orders on a normal day. "
      f"The table shows the four largest losses across {len(disruptions)} individual outages.")
display(disruptions.head(4).set_index(["Type", "Outage at"]).round(1))

Current capacity supports 2,291 orders on a normal day. The table shows the four largest losses across 28 individual outages.


Orders during outage  Drop in daily orders  \
Type      Outage at                                               
Warehouse PLANT03                    1278                  1013   
Carrier   V444_5                     1790                   501   
Warehouse PLANT04                    1916                   375   
          PLANT12                    2082                   209   

                     Throughput loss (%)  
Type      Outage at                       
Warehouse PLANT03                   44.2  
Carrier   V444_5                    21.9  
Warehouse PLANT04                   16.4  
          PLANT12                    9.1

### 3.2 Which expansion plan maximizes expected throughput under outage risk?

Choose up to two warehouses before the outage is known. Each selected warehouse gains 100 slots per day. Evaluate every plan on a normal day and under the same 28 individual outages. Assign orders using the network available in each scenario.

The data do not give outage frequencies. We assume that the 28 scenarios are equally likely conditional on an outage. If $p$ is the probability of an outage on a given day, expected daily throughput is

$$(1-p)\times\text{normal-day throughput}+p\times\text{mean throughput on outage days}.$$

In [7]:
from itertools import combinations

choices = [sites_to_expand for k in range(SITE_LIMIT + 1) for sites_to_expand in combinations(plants, k)]
plan_caps = pd.DataFrame([capacity + EXTRA * capacity.index.isin(choice) for choice in choices],
                         index=[" + ".join(choice) or "Current capacity" for choice in choices], columns=plants)
plan_caps.index.name = "Expansion sites"

def evaluate_capacity_plans(options, plans):
    # Each order uses one slot. Grouping orders with identical eligible warehouses preserves maximum throughput.
    groups = options.groupby("order", sort=False)["plant"].agg(lambda ws: tuple(sorted(ws))).value_counts()
    with gp.Model(env=env) as model:
        incoming = {w: [] for w in plans.columns}
        for warehouses, n_orders in groups.items():
            flow = model.addVars(warehouses, obj=1)
            model.addConstr(flow.sum() <= int(n_orders))
            for w in warehouses:
                incoming[w].append(flow[w])
        limits = [model.addConstr(gp.quicksum(incoming[w]) <= 0) for w in plans.columns]
        model.ModelSense = GRB.MAXIMIZE
        totals = []
        for rhs in plans.itertuples(index=False, name=None):
            model.setAttr("RHS", limits, rhs)
            model.optimize()
            if model.Status != GRB.OPTIMAL:
                raise RuntimeError(f"The throughput model did not reach an optimum (status {model.Status}).")
            totals.append(round(model.ObjVal))
    return totals

# Rows represent expansion plans chosen in advance. Columns represent operating scenarios.
plan_throughput = pd.DataFrame({name: evaluate_capacity_plans(options, plan_caps)
                                for name, options in scenario_options.items()}, index=plan_caps.index)
outage_names = disruptions["Outage at"].to_list()
assert plan_throughput.at["Current capacity", "Normal operations"] == N0
assert plan_throughput.loc["Current capacity", outage_names].to_list() == disruptions["Orders during outage"].to_list()
assert plan_throughput.at[sites_text, "Normal operations"] == expanded["maximum"]

In [8]:
normal_day = plan_throughput["Normal operations"]
outage_mean = plan_throughput[outage_names].mean(axis=1)

def expected_orders(p_outage):
    """Return expected daily throughput for each plan at the given outage probability."""
    if not 0 <= p_outage <= 1:
        raise ValueError("The outage probability must be between 0 and 1, inclusive.")
    return (1 - p_outage) * normal_day + p_outage * outage_mean

risk_comparison = pd.DataFrame({"Normal-day orders": normal_day, "Mean orders on outage days": expected_orders(1)})
risk_comparison["Mean outage-day gain over current capacity"] = outage_mean - outage_mean["Current capacity"]
ranked = risk_comparison.sort_values(["Mean orders on outage days", "Normal-day orders"], ascending=False)
print(f"We compared {len(plan_caps)} plans. The table shows current capacity and the three expansions "
      "with the highest mean throughput on outage days.")
display(pd.concat([ranked.loc[["Current capacity"]], ranked.drop(index="Current capacity").head(3)]).round(2))

# A plan that leads on both measures maximizes their weighted average for any outage probability.
best_in_both = normal_day.index[normal_day.eq(normal_day.max()) & outage_mean.eq(outage_mean.max())]
if len(best_in_both):
    for plan in best_in_both:
        print(f"{plan} ranks first with {normal_day[plan]:,} orders on a normal day "
              f"and {outage_mean[plan]:,.2f} on average on an outage day. "
              "With equally likely outages, this plan maximizes expected throughput for any probability of an outage day.")
else:
    print("The best plans differ by day type. For an outage probability p, "
          "use expected_orders(p).idxmax() to select a plan.")

# Which sites maximize throughput if the warehouse with the largest baseline loss closes?
critical_plant = disruptions.loc[disruptions["Type"].eq("Warehouse"), "Outage at"].iloc[0]
stress_day = plan_throughput[critical_plant]
best_stress = " or ".join(stress_day.index[stress_day.eq(stress_day.max())])
print(f"If {critical_plant} closes, the maximum is {stress_day.max():,} orders. "
      f"Expanding {best_stress} achieves this result. "
      f"The Question 2 plan ({sites_text}) can handle {stress_day[sites_text]:,} orders.")

We compared 191 plans. The table shows current capacity and the three expansions with the highest mean throughput on outage days.


,Normal-day orders,Mean orders on outage days,Mean outage-day gain over current capacity
Expansion sites,,,
Current capacity,2291,2185.89,0.00
PLANT02 + PLANT03,2491,2378.75,192.86
PLANT02 + PLANT12,2456,2344.18,158.29
PLANT03 + PLANT12,2456,2344.18,158.29


PLANT02 + PLANT03 ranks first with 2,491 orders on a normal day and 2,378.75 on average on an outage day. With equally likely outages, this plan maximizes expected throughput for any probability of an outage day.
If PLANT03 closes, the maximum is 1,443 orders. Expanding PLANT02 + PLANT10 or PLANT02 + PLANT12 achieves this result. The Question 2 plan (PLANT02 + PLANT03) can handle 1,378 orders.
